# 4.2 Variational Autoencoder (II): Generating new samples
***

## Load weights

In [ ]:
## Import modules
import torch

## Load weight parameters and some metadata
FNAME_MODEL = 'models/vae.pt'
checkpoint = torch.load(FNAME_MODEL)

## Restore trained model

In [ ]:
from helper import VariationalAutoencoder

### Recreate the model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'])
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
# Load model for inference
model.eval()

## Generating new samples

In [ ]:
from helper import MinMaxScale

## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
transform = MinMaxScale(min_value, max_value)

In [ ]:
## Generate nens new samples
nens = 1064
z = torch.randn(nens, checkpoint['LATENT_DIM'])
with torch.no_grad():
    new_sample = model.decode(z)
    x = transform.invert(new_sample).squeeze()
print(z.shape)
print(x.shape)

## Plotting new samples

In [ ]:
import matplotlib.pyplot as plt

## Plotting new samples
plot_conf = {
    'cmap': 'RdYlBu_r',
    'vmin': 0, 
    'vmax': 30,
}

fig, axs = plt.subplots(nrows=4, ncols=3, figsize=(8,10))

for i, ax in enumerate(axs.flat):
    ax.pcolormesh(x[i],**plot_conf)
    ax.set_xticks([])
    ax.set_yticks([])

## Exceedance probabilities

In [ ]:
## Compute the probability of exceeding 10 g/m2
threshold = 10
probabilities = 100 * (x > threshold).float().mean(dim=0)
probabilities.max()

In [ ]:
from helper_plot import create_map
#import matplotlib.pyplot as plt

## Create a probability contour plot
fig, ax = create_map()
#fig, ax = plt.subplots()

lat = [22. , 22.1, 22.2, 22.3, 22.4, 22.5, 22.6, 22.7, 22.8, 22.9, 23. , 23.1,
       23.2, 23.3, 23.4, 23.5, 23.6, 23.7, 23.8, 23.9, 24. , 24.1, 24.2, 24.3,
       24.4, 24.5, 24.6, 24.7, 24.8, 24.9, 25. , 25.1, 25.2, 25.3, 25.4, 25.5,
       25.6, 25.7, 25.8, 25.9, 26. , 26.1, 26.2, 26.3, 26.4, 26.5, 26.6, 26.7,
       26.8, 26.9, 27. , 27.1, 27.2, 27.3, 27.4, 27.5, 27.6, 27.7, 27.8, 27.9,
       28. , 28.1, 28.2, 28.3, 28.4, 28.5, 28.6, 28.7, 28.8, 28.9, 29. , 29.1,
       29.2, 29.3, 29.4, 29.5, 29.6, 29.7, 29.8, 29.9, 30. , 30.1, 30.2, 30.3,
       30.4, 30.5, 30.6, 30.7, 30.8, 30.9, 31. , 31.1, 31.2, 31.3, 31.4, 31.5,
       31.6, 31.7, 31.8, 31.9, 32. ]

lon = [-22. , -21.9, -21.8, -21.7, -21.6, -21.5, -21.4, -21.3, -21.2, -21.1,
       -21. , -20.9, -20.8, -20.7, -20.6, -20.5, -20.4, -20.3, -20.2, -20.1,
       -20. , -19.9, -19.8, -19.7, -19.6, -19.5, -19.4, -19.3, -19.2, -19.1,
       -19. , -18.9, -18.8, -18.7, -18.6, -18.5, -18.4, -18.3, -18.2, -18.1,
       -18. , -17.9, -17.8, -17.7, -17.6, -17.5, -17.4, -17.3, -17.2, -17.1,
       -17. , -16.9, -16.8, -16.7, -16.6, -16.5, -16.4, -16.3, -16.2, -16.1,
       -16. , -15.9, -15.8, -15.7, -15.6, -15.5, -15.4, -15.3, -15.2, -15.1,
       -15. , -14.9, -14.8, -14.7, -14.6, -14.5, -14.4, -14.3, -14.2, -14.1,
       -14. , -13.9, -13.8, -13.7, -13.6, -13.5, -13.4, -13.3, -13.2, -13.1,
       -13. , -12.9, -12.8, -12.7, -12.6, -12.5, -12.4, -12.3, -12.2, -12.1,
       -12. , -11.9, -11.8, -11.7, -11.6, -11.5, -11.4, -11.3, -11.2, -11.1,
       -11. , -10.9, -10.8, -10.7, -10.6, -10.5, -10.4, -10.3, -10.2, -10.1,
       -10. ]

cs = ax.contour(
    lon,lat,probabilities,
    levels    = [2, 25, 50, 75, 98],
    )
ax.clabel(cs, cs.levels, inline = False, fontsize=10)
ax.set_extent([-22, -11, 23.5, 29]) # [x1,x2,y1,y2]
ax.set(title='Exceedance probabilities')

## Ensemble mean

In [ ]:
fig, ax = create_map()

fc = ax.contourf(lon,lat,
                 x.mean(dim=0),
                 levels = [5,10,15,20,25,30,35,40],
                )

cbar = fig.colorbar(fc)